# TRL SFT 内部实现：从数据到参数更新

这份 Notebook 不把 `SFTTrainer` 当作黑盒。我们会自己实现它背后最关键的链路：

1. 对话模板：`messages -> text`
2. 分词：`text -> input_ids`
3. 标签掩码：只训练 completion 或 assistant
4. 动态 padding：`labels` 使用 `-100`
5. 因果语言模型前向传播：得到 `[B, T, V]` logits
6. next-token shift 与 token-level cross entropy
7. `backward -> optimizer.step`
8. 最后再与 TRL `SFTTrainer` 对照

> 默认部分只需要 PyTorch，不依赖 TRL，也不下载模型。这里的小模型只用于暴露训练机制；换成 Qwen/Llama 后，SFT 数据、标签和损失原理不变。

## 0. 对照资料与源码入口

- [TRL SFTTrainer 中文文档](https://hugging-face.cn/docs/trl/sft_trainer)
- [TRL SFT 原理说明](https://github.com/huggingface/trl/blob/main/docs/source/sft_trainer.md#looking-deeper-into-the-sft-method)
- [TRL `sft_trainer.py`](https://github.com/huggingface/trl/blob/main/trl/trainer/sft_trainer.py)
- [Transformers `trainer.py`](https://github.com/huggingface/transformers/blob/main/src/transformers/trainer.py)

阅读源码时，把本 Notebook 中的函数与框架实现对应起来：

| 本 Notebook | 框架中对应的职责 |
|---|---|
| `apply_chat_template_manual` | chat template / `apply_chat_template` |
| `preprocess_prompt_completion` | TRL 的数据集准备与 tokenize |
| `preprocess_messages_assistant_only` | `assistant_only_loss` 标签掩码 |
| `DataCollatorForCausalSFT` | data collator / padding |
| `causal_lm_loss` | 模型损失或 Trainer 的 loss 计算 |
| `train_one_step` | `Trainer.training_step` 的核心数学过程 |

In [1]:
import math
import random
import re
from collections import Counter
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("训练设备:", device)


PyTorch: 2.11.0+cu128
训练设备: cuda


SFT 支持语言建模 (language modeling) 和提示-补全 (prompt-completion) 数据集。SFTTrainer 兼容标准和对话式数据集格式。当提供对话式数据集时，训练器会自动将聊天模板应用于数据集。

In [2]:
# Standard language modeling
{"text": "The sky is blue."}

# Conversational language modeling
{"messages": [{"role": "user", "content": "What color is the sky?"},
              {"role": "assistant", "content": "It is blue."}]}

# Standard prompt-completion
{"prompt": "The sky is",
 "completion": " blue."}

# Conversational prompt-completion
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "completion": [{"role": "assistant", "content": "It is blue."}]}

{'prompt': [{'role': 'user', 'content': 'What color is the sky?'}],
 'completion': [{'role': 'assistant', 'content': 'It is blue.'}]}

## 1. 准备一个最小 prompt-completion 数据集

TRL 支持 `text`、`prompt/completion` 和 `messages` 等格式。我们先使用 prompt-completion，因为它最容易看清楚：prompt token 可以被设为 `-100`，只让 completion token 产生 loss。

In [2]:
raw_examples = [
    {"prompt": "1 加 1 等于多少？", "completion": "2。"},
    {"prompt": "中国的首都是哪里？", "completion": "北京。"},
    {"prompt": "天空通常是什么颜色？", "completion": "蓝色。"},
    {"prompt": "水在标准气压下多少摄氏度沸腾？", "completion": "100 摄氏度。"},
]

conversation_example = {
    "messages": [
        {"role": "system", "content": "你是一个简洁的问答助手。"},
        {"role": "user", "content": "法国的首都是哪里？"},
        {"role": "assistant", "content": "巴黎。"},
        {"role": "user", "content": "它位于哪条河附近？"},
        {"role": "assistant", "content": "塞纳河。"},
    ]
}

raw_examples[0], conversation_example


({'prompt': '1 加 1 等于多少？', 'completion': '2。'},
 {'messages': [{'role': 'system', 'content': '你是一个简洁的问答助手。'},
   {'role': 'user', 'content': '法国的首都是哪里？'},
   {'role': 'assistant', 'content': '巴黎。'},
   {'role': 'user', 'content': '它位于哪条河附近？'},
   {'role': 'assistant', 'content': '塞纳河。'}]})

## 2. 手写 Chat Template

真实模型的模板一般存放在 tokenizer 中。这里用一个简化的 ChatML 风格模板，把角色和内容变成一个序列。注意：模板不是装饰，它决定了模型实际看到哪些 token，也决定 assistant 区间从哪里开始。

In [ ]:
ROLE_TOKEN = {
    "system": "<|system|>",
    "user": "<|user|>",
    "assistant": "<|assistant|>",
}
END_TOKEN = "<|end|>"

def apply_chat_template_manual(messages, add_generation_prompt=False):
    """把结构化 messages 转成模型真正读取的字符串。"""
    pieces = []
    for message in messages:
        role = message["role"]
        if role not in ROLE_TOKEN:
            raise ValueError(f"不支持的角色: {role}")
        pieces.extend([ROLE_TOKEN[role], message["content"], END_TOKEN])

    if add_generation_prompt:
        pieces.append(ROLE_TOKEN["assistant"])

    return " ".join(pieces)


rendered = apply_chat_template_manual(
    conversation_example["messages"],
    add_generation_prompt=False,
)
print(rendered)


<|system|> 你是一个简洁的问答助手。 <|end|> <|user|> 法国的首都是哪里？ <|end|> <|assistant|> 巴黎。 <|end|> <|user|> 它位于哪条河附近？ <|end|> <|assistant|> 塞纳河。 <|end|>


## 3. 手写一个最小 tokenizer

它不是 BPE/SentencePiece 的复现，而是一个用于观察 `input_ids`、padding 和 labels 的透明 tokenizer。真实项目中只需把它换成 `AutoTokenizer`，后面的 SFT 流程不变。

In [4]:
TOKEN_PATTERN = re.compile(r"<\|[^|]+\|>|[A-Za-z]+|\d+|[\u4e00-\u9fff]|[^\s]")


class SimpleTokenizer:
    def __init__(self, texts):
        special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
        tokens = []
        for text in texts:
            tokens.extend(self.tokenize(text))

        vocab_tokens = special_tokens + sorted(set(tokens) - set(special_tokens))
        self.token_to_id = {token: idx for idx, token in enumerate(vocab_tokens)}
        self.id_to_token = {idx: token for token, idx in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.unk_token_id = self.token_to_id["<unk>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]

    def __len__(self):
        return len(self.token_to_id)

    def tokenize(self, text):
        return TOKEN_PATTERN.findall(text)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.token_to_id.get(token, self.unk_token_id) for token in self.tokenize(text)]
        if add_bos:
            ids = [self.bos_token_id] + ids
        if add_eos:
            ids = ids + [self.eos_token_id]
        return ids

    def convert_ids_to_tokens(self, ids):
        return [self.id_to_token[int(idx)] for idx in ids]

    def decode(self, ids, skip_special_tokens=False):
        tokens = self.convert_ids_to_tokens(ids)
        if skip_special_tokens:
            tokens = [t for t in tokens if not (t.startswith("<") and t.endswith(">"))]
        return " ".join(tokens)


corpus = []
for example in raw_examples:
    corpus.extend([example["prompt"], example["completion"]])
corpus.append(rendered)
tokenizer = SimpleTokenizer(corpus)

demo_ids = tokenizer.encode("中国的首都是哪里？", add_bos=True, add_eos=True)
print("词表大小:", len(tokenizer))
print("ids:", demo_ids)
print("tokens:", tokenizer.convert_ids_to_tokens(demo_ids))


词表大小: 69
ids: [2, 15, 28, 50, 66, 61, 40, 27, 62, 68, 3]
tokens: ['<bos>', '中', '国', '的', '首', '都', '是', '哪', '里', '？', '<eos>']


## 4. 手写 prompt-completion 预处理与 labels mask

`input_ids` 是模型输入；`labels` 是每个位置的监督目标。设置 `completion_only_loss=True` 时，prompt 对应的位置写成 `-100`。PyTorch 的交叉熵会忽略这些位置。

这里分别 tokenize prompt 和 completion，是为了精确知道边界。真实 tokenizer 可能存在边界合并行为，因此生产实现还要仔细处理拼接前后 tokenize 不一致的问题。

In [ ]:
# 这里 completion only loss 个地方的实现，就是说如果说我只计算它的一个完成度的话，那么我的 label 除了 completion 部分之外，都要设置成 -100。
def preprocess_prompt_completion(
    example,
    tokenizer,
    max_length=128,
    completion_only_loss=True,
):
    """复现 prompt-completion SFT 的 input_ids、attention_mask 和 labels 构造。"""
    prompt_ids = tokenizer.encode(example["prompt"], add_bos=True, add_eos=False)
    completion_ids = tokenizer.encode(example["completion"], add_bos=False, add_eos=True)

    # Padding部分为0，然后非Padding部分为1
    input_ids = (prompt_ids + completion_ids)[:max_length]
    attention_mask = [1] * len(input_ids)

    # 计算 loss 的部分用 -100
    if completion_only_loss:
        prompt_length = min(len(prompt_ids), len(input_ids))
        labels = [-100] * prompt_length + input_ids[prompt_length:]
    else:
        labels = input_ids.copy()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


encoded_example = preprocess_prompt_completion(raw_examples[0], tokenizer)
encoded_example


{'input_ids': [2, 4, 23, 4, 52, 17, 31, 34, 68, 6, 11, 3],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, -100, 6, 11, 3]}

In [6]:
def inspect_supervision(encoded, tokenizer):
    """逐 token 显示哪些位置参与 loss。"""
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
    rows = []
    for position, (token, label) in enumerate(zip(tokens, encoded["labels"])):
        target = "忽略" if label == -100 else tokenizer.id_to_token[label]
        rows.append((position, token, label, target))
    return rows


print("位置 | 输入 token | label id | 是否监督")
for row in inspect_supervision(encoded_example, tokenizer):
    print(f"{row[0]:>4} | {row[1]:>10} | {row[2]:>8} | {row[3]}")


位置 | 输入 token | label id | 是否监督
   0 |      <bos> |     -100 | 忽略
   1 |          1 |     -100 | 忽略
   2 |          加 |     -100 | 忽略
   3 |          1 |     -100 | 忽略
   4 |          等 |     -100 | 忽略
   5 |          于 |     -100 | 忽略
   6 |          多 |     -100 | 忽略
   7 |          少 |     -100 | 忽略
   8 |          ？ |     -100 | 忽略
   9 |          2 |        6 | 2
  10 |          。 |       11 | 。
  11 |      <eos> |        3 | <eos>


## 5. 手写 assistant-only labels

多轮对话中，`assistant_only_loss=True` 的关键不是“删除 user”，而是 user/system token 仍作为上下文输入模型，只把它们的 label 设成 `-100`。下面的函数保留完整对话，但只监督 assistant 内容和结束 token。

In [7]:
def preprocess_messages_assistant_only(messages, tokenizer, max_length=256):
    """把多轮 messages 编码为完整上下文，但只让 assistant 区间产生 loss。"""
    input_ids = [tokenizer.bos_token_id]
    labels = [-100]

    for message in messages:
        role = message["role"]
        role_ids = tokenizer.encode(ROLE_TOKEN[role])
        content_ids = tokenizer.encode(message["content"])
        end_ids = tokenizer.encode(END_TOKEN)

        segment_ids = role_ids + content_ids + end_ids
        input_ids.extend(segment_ids)

        if role == "assistant":
            # 角色标记只用于提供上下文；监督 assistant 的内容及结束 token。
            labels.extend([-100] * len(role_ids) + content_ids + end_ids)
        else:
            labels.extend([-100] * len(segment_ids))

    input_ids = input_ids[:max_length]
    labels = labels[:max_length]
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


assistant_encoded = preprocess_messages_assistant_only(
    conversation_example["messages"], tokenizer
)
for row in inspect_supervision(assistant_encoded, tokenizer):
    print(f"{row[0]:>4} | {row[1]:>13} | {row[2]:>8} | {row[3]}")


   0 |         <bos> |     -100 | 忽略
   1 |    <|system|> |     -100 | 忽略
   2 |             你 |     -100 | 忽略
   3 |             是 |     -100 | 忽略
   4 |             一 |     -100 | 忽略
   5 |             个 |     -100 | 忽略
   6 |             简 |     -100 | 忽略
   7 |             洁 |     -100 | 忽略
   8 |             的 |     -100 | 忽略
   9 |             问 |     -100 | 忽略
  10 |             答 |     -100 | 忽略
  11 |             助 |     -100 | 忽略
  12 |             手 |     -100 | 忽略
  13 |             。 |     -100 | 忽略
  14 |       <|end|> |     -100 | 忽略
  15 |      <|user|> |     -100 | 忽略
  16 |             法 |     -100 | 忽略
  17 |             国 |     -100 | 忽略
  18 |             的 |     -100 | 忽略
  19 |             首 |     -100 | 忽略
  20 |             都 |     -100 | 忽略
  21 |             是 |     -100 | 忽略
  22 |             哪 |     -100 | 忽略
  23 |             里 |     -100 | 忽略
  24 |             ？ |     -100 | 忽略
  25 |       <|end|> |     -100 | 忽略
  26 | <|assistant|> |     -100 | 忽略
 

## 6. 手写动态 padding collator

同一个 batch 的序列长度必须一致：

- `input_ids` 补 `pad_token_id`
- `attention_mask` 补 `0`
- `labels` 补 `-100`

因此 padding token 不会参与注意力，也不会产生 loss。

In [ ]:
@dataclass
class DataCollatorForCausalSFT:
    pad_token_id: int
    label_pad_token_id: int = -100

    def __call__(self, features):
        max_length = max(len(feature["input_ids"]) for feature in features)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            pad_length = max_length - len(feature["input_ids"])
            batch_input_ids.append(
                feature["input_ids"] + [self.pad_token_id] * pad_length
            )
            batch_attention_mask.append(
                feature["attention_mask"] + [0] * pad_length
            )
            batch_labels.append(
                feature["labels"] + [self.label_pad_token_id] * pad_length
            )

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }


collator = DataCollatorForCausalSFT(tokenizer.pad_token_id)
features = [preprocess_prompt_completion(x, tokenizer) for x in raw_examples[:2]]
demo_batch = collator(features)
for key, value in demo_batch.items():
    print(key, value.shape)
    print(value)


input_ids torch.Size([2, 14])
tensor([[ 2,  4, 23,  4, 52, 17, 31, 34, 68,  6, 11,  3,  0,  0],
        [ 2, 15, 28, 50, 66, 61, 40, 27, 62, 68, 25, 18, 11,  3]])
attention_mask torch.Size([2, 14])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
labels torch.Size([2, 14])
tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100,    6,   11,    3,
         -100, -100],
        [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100,   25,   18,
           11,    3]])


## 7. 构造一个最小因果语言模型

为了让 Notebook 完全自包含，这里使用一个很小的 Transformer。真正的 Qwen/Llama 结构复杂得多，但输出仍然是 `[batch, sequence, vocab]` 的 logits。

上三角 causal mask 保证位置 `t` 不能偷看未来 token。

In [9]:
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, hidden_size=64, num_heads=4, num_layers=2, max_length=256):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.position_embedding = nn.Embedding(max_length, hidden_size)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=0.0,
            batch_first=True,
            norm_first=False,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.final_norm = nn.LayerNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_length = input_ids.shape
        positions = torch.arange(seq_length, device=input_ids.device).unsqueeze(0)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)

        # True 表示该注意力位置被屏蔽。
        causal_mask = torch.triu(
            torch.ones(seq_length, seq_length, dtype=torch.bool, device=input_ids.device),
            diagonal=1,
        )
        padding_mask = None if attention_mask is None else attention_mask.eq(0)

        hidden = self.transformer(
            hidden,
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        hidden = self.final_norm(hidden)
        return self.lm_head(hidden)


model = TinyCausalLM(len(tokenizer)).to(device)
with torch.no_grad():
    logits = model(
        demo_batch["input_ids"].to(device),
        demo_batch["attention_mask"].to(device),
    )
print("input_ids shape:", demo_batch["input_ids"].shape)
print("logits shape:", logits.shape, "= [batch, sequence, vocab]")


input_ids shape: torch.Size([2, 14])
logits shape: torch.Size([2, 14, 69]) = [batch, sequence, vocab]


## 8. 手写 next-token SFT loss

因果语言模型在位置 `t` 的 logits 预测位置 `t+1` 的 token，因此需要错位：

```python
shift_logits = logits[:, :-1, :]
shift_labels = labels[:, 1:]
```

`ignore_index=-100` 同时忽略 prompt、非 assistant 消息和 padding。

In [10]:
def causal_lm_loss(logits, labels, ignore_index=-100):
    """标准 token-level NLL / cross entropy，与 Causal LM SFT 的核心数学一致。"""
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    vocab_size = shift_logits.size(-1)
    loss = F.cross_entropy(
        shift_logits.view(-1, vocab_size),
        shift_labels.view(-1),
        ignore_index=ignore_index,
        reduction="mean",
    )
    return loss


batch_on_device = {key: value.to(device) for key, value in demo_batch.items()}
logits = model(batch_on_device["input_ids"], batch_on_device["attention_mask"])
loss = causal_lm_loss(logits, batch_on_device["labels"])
print("初始 loss:", float(loss))
print("随机模型的理论基线 log(vocab_size):", math.log(len(tokenizer)))


初始 loss: 4.315553665161133
随机模型的理论基线 log(vocab_size): 4.23410650459726


C:\Users\bbfss\AppData\Local\Temp\ipykernel_38700\1562962840.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print("初始 loss:", float(loss))


### 8.1 再拆一层：不用 `F.cross_entropy` 验证公式

交叉熵在 one-hot 标签下等价于：对正确 token 的 `log_softmax` 取负号，再对有效 token 求平均。下面手算一次，并与 PyTorch 结果比较。

In [ ]:
def causal_lm_loss_manual(logits, labels, ignore_index=-100):
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    valid_mask = shift_labels.ne(ignore_index) # 有效为1，无效为0

    safe_labels = shift_labels.masked_fill(~valid_mask, 0) # 无效值部分填充为 0
    log_probs = F.log_softmax(shift_logits, dim=-1)
    
    # target_log_probs = [batch_size, seq_len, vocab_size]， safe_labels.unsqueeze(-1)  = [batch_size, seq_len， 1] ， 从模型输出的所有词的概率分布中，精准提取出“真实目标词”对应的概率（或对数概率）
    target_log_probs = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    token_losses = -target_log_probs
    return token_losses[valid_mask].mean()


loss_builtin = causal_lm_loss(logits, batch_on_device["labels"])
loss_manual = causal_lm_loss_manual(logits, batch_on_device["labels"])
print("F.cross_entropy:", float(loss_builtin))
print("手算 NLL:", float(loss_manual))
print("是否一致:", torch.allclose(loss_builtin, loss_manual, atol=1e-6))


F.cross_entropy: 4.315553665161133
手算 NLL: 4.315553665161133
是否一致: True


## 9. 手写一个完整 training step

这就是 `Trainer.training_step` 最核心的部分。实际 Trainer 还会加入混合精度、梯度累积、分布式同步、日志、checkpoint 和 scheduler。

In [12]:
def train_one_step(model, batch, optimizer, max_grad_norm=1.0):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits = model(batch["input_ids"], batch["attention_mask"])
    loss = causal_lm_loss(logits, batch["labels"])
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    optimizer.step()

    return float(loss.detach()), float(grad_norm)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
before = model.lm_head.weight.detach().clone()
step_loss, grad_norm = train_one_step(model, batch_on_device, optimizer)
after = model.lm_head.weight.detach().clone()

print("loss:", step_loss)
print("梯度范数:", grad_norm)
print("参数是否更新:", not torch.equal(before, after))


loss: 4.315553665161133
梯度范数: 3.994694232940674
参数是否更新: True


## 10. 手写完整训练循环

数据很小，目标只是验证 loss 能下降。不要把这里的超参数当成真实大模型训练配置。

In [13]:
class EncodedSFTDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.features = [
            preprocess_prompt_completion(example, tokenizer)
            for example in examples
        ]

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]


train_dataset = EncodedSFTDataset(raw_examples, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

model = TinyCausalLM(len(tokenizer)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)

for epoch in range(30):
    epoch_losses = []
    for batch in train_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        step_loss, _ = train_one_step(model, batch, optimizer)
        epoch_losses.append(step_loss)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch={epoch + 1:02d}  loss={sum(epoch_losses) / len(epoch_losses):.4f}")


epoch=01  loss=4.1072
epoch=05  loss=0.5868
epoch=10  loss=0.0893
epoch=15  loss=0.0277
epoch=20  loss=0.0131
epoch=25  loss=0.0082
epoch=30  loss=0.0063


## 11. 与 TRL `SFTTrainer` 对照（可选）

当前 `verl` Conda 环境已经安装 `trl 1.8.0`、`datasets` 和 `peft`。如果以后在新环境中运行，可使用：

```powershell
E:\anaconda3\envs\verl\python.exe -m pip install -U trl datasets peft
```

安装后重启 Notebook 内核。下面默认 `RUN_TRL_DEMO=False`，避免意外下载模型。

In [14]:
RUN_TRL_DEMO = False

if RUN_TRL_DEMO:
    from datasets import Dataset as HFDataset
    from trl import SFTConfig, SFTTrainer

    hf_dataset = HFDataset.from_list(raw_examples)
    args = SFTConfig(
        output_dir="./outputs/trl-sft-demo",
        max_length=128,
        completion_only_loss=True,
        per_device_train_batch_size=2,
        num_train_epochs=1,
        learning_rate=2e-5,
        logging_steps=1,
        report_to="none",
    )

    trainer = SFTTrainer(
        model="Qwen/Qwen2.5-0.5B-Instruct",
        args=args,
        train_dataset=hf_dataset,
    )
    trainer.train()
else:
    print("TRL 对照实验未运行。理解前面 1～10 节后再开启。")


TRL 对照实验未运行。理解前面 1～10 节后再开启。


## 12. 接下来如何读 TRL 与 verl 源码

现在再去看框架源码，并始终问一句：“这段代码对应我刚才手写的哪一步？”

### TRL 阅读顺序

1. `trl/trainer/sft_trainer.py` 中的数据集准备与 tokenize
2. labels 在哪里被设置成 `-100`
3. completion-only 与 assistant-only 的 mask 从哪里产生
4. `SFTTrainer` 哪些行为来自父类 `transformers.Trainer`
5. `transformers/trainer.py` 的 `training_step` 和完整训练循环

### 映射到 verl

1. 先找 verl SFT batch 中的 `input_ids`、`attention_mask`、`labels` 或 loss mask
2. 再找 logits 如何 shift
3. 再找 token loss 如何被 mask 和归一化
4. 最后才看 FSDP、device mesh、checkpoint 和分布式通信

verl 增加的主要是大规模训练工程能力；SFT 的数学核心仍是本 Notebook 第 8 节的 next-token NLL。

## 学习检查点

运行并理解 Notebook 后，先尝试用自己的话回答：

1. 为什么 prompt token 仍存在于 `input_ids`，却可以不产生 loss？
2. 为什么是 `logits[:, :-1]` 对齐 `labels[:, 1:]`？
3. padding、user token 和 system token 都可能写成 `-100`，但三者被忽略的原因分别是什么？
4. 如果所有 labels 都是 `-100`，当前 loss 会发生什么？应该怎样在数据处理中防止？
5. 把小模型替换成 Qwen 后，哪些函数仍然可以保持不变？

不要急着继续到 FSDP。能结合一个具体 batch 解释这五个问题，才算真正掌握 SFT 的核心链路。

# VERL - SFT 内部实现复现

## 数据处理 - 生成loss mask 和 attention mask 和 分词

在数据预处理阶段， 主要是生成inputs id， 然后对每个message生成 
1. inputs id， 这里tokneize = True 就是在进行分词 
2. attention mask ，这里就是确实是prompt部分的分词就设置为1， 然后PAD部分就为0（为了让长度一致）
3. loss mask ： 除了assistant的回答之外的部分全部为0，不希望模型进行学习， 然后回答的部分为1， 希望学习生成回答的部分  

In [ ]:
class MultiTurnSFTDataset(Dataset):
    """
    Dataset for multi-turn conversations where each assistant response should be trained

    Args:
        data_files (str or list): Path(s) to Parquet file(s).
        tokenizer (PreTrainedTokenizer): For the tokenization of text to token IDs.
        config (DictConfig): Options like cache_dir, prompt_key, max_prompt_length, truncation, etc.
        processor (ProcessorMixin, optional): Multimodal preprocessor for images/videos.
        max_samples (int, optional): Limit the number of samples. Defaults to -1 (use all).
    """
    def _process_single_message(
            self,
            index: int,
            message: dict[str, Any],
            full_message: list,
            tools: Optional[list[dict[str, Any]]] = None,
            enable_thinking: Optional[bool] = None,
        ) -> tuple[list[int], list[int], list[int]]:
            """
            Process a single message and return its tokenized representation.

            Args:
                index: turn index in the conversation
                message: A single message dictionary
                images: List of images to be used
                videos: List of videos to be used
                tools: List of tools to be used
                enable_thinking: Whether to enable thinking mode

            Returns:
                Tuple of (input_ids, loss_mask, attention_mask, dict[str, torch.Tensor])
            """
            processor = self.processor if self.processor is not None else self.tokenizer
            apply_chat_template_kwargs = {**self.apply_chat_template_kwargs}
            if enable_thinking is not None:
                apply_chat_template_kwargs["enable_thinking"] = enable_thinking

            inputs = apply_chat_template(
                processor,
                messages=[message],
                tools=tools,
                add_generation_prompt=False,
                tokenize=True, # 进行分词，生成inputs id 
                return_dict=True,   # 这里会同时生成attenion mask 然后同时返回一个dict
                return_tensors="pt",
                **apply_chat_template_kwargs,
            )

            inputs = dict(inputs) # 把特殊的BatchEncoding对象转成普通dict，方便后续处理
            input_ids = inputs.pop("input_ids")[0]
            attention_mask = inputs.pop("attention_mask")[0]


            # 原因是 _process_single_message() 会把每一条消息单独送进 apply_chat_template()：某些 tokenizer 在处理单条消息时，会自动在前面补一个默认 system prompt。于是分别处理时可能变成：
            #messages = [
            # {"role": "system", "content": "你是一个助手"},  # index = 0
            # {"role": "user", "content": "你好"},            # index = 1
            # {"role": "assistant", "content": "你好！"},      # index = 2
            # {"role": "user", "content": "介绍一下北京"},     # index = 3
            # {"role": "assistant", "content": "北京是……"},    # index = 4
            # ]
            # 
            # 变成下面
            # 
            # 第0条：system prompt
            # 第1条：默认 system prompt + user prompt
            # 第2条：默认 system prompt + assistant prompt
            # 第3条：默认 system prompt + user prompt
            # remove system prompt if exists
            if index != 0 and message["role"] != "system":
                input_ids = input_ids[len(self.system_prompt) :]
                attention_mask = attention_mask[len(self.system_prompt) :]

            # loss mask 为 1 代表要计算loss， 0代表不计算loss， 这里逻辑是assistant message 才计算loss，其他角色的消息不计算loss，同时要屏蔽掉生成 prompt 的部分：所以loss_mask[: len(self.generation_prompt)] = 0
            if message["role"] == "assistant":
                loss_mask = torch.ones_like(attention_mask)
                # mask out generation prompt if assistant message
                loss_mask[: len(self.generation_prompt)] = 0
            else:
                loss_mask = torch.zeros_like(attention_mask)

            return input_ids, loss_mask, attention_mask, inputs

## SFT loss - 只计算生成response部分的结果

SFT loss 会生成一个loss mask （在padding 模式下叫response mask），然后他会做移动一位，然后和log prob 进行一个交叉熵， 然后计算出来的结果就是生成出来的结果的交叉熵结果

这里
labels=-100 → cross_entropy 自动忽略

verl：
loss_mask=0 → masked_sum 手动忽略

In [ ]:
def sft_loss(config: ActorConfig, model_output, data: TensorDict, dp_group=None):
    pad_mode = tu.get_non_tensor_data(data=data, key="pad_mode", default=DatasetPadMode.NO_PADDING)
    dp_size = data["dp_size"]
    batch_num_tokens = data["batch_num_tokens"]

    log_prob = model_output["log_probs"]


    # 当数据没有使用 Padding 时，通常意味着多条变长文本被打包在一起
    # mask 左移一位（shifts=-1）→ 实现 预测 token t 与 真值 token t+1 的 mask 一一对应，损失计算合法有效。
    if pad_mode == DatasetPadMode.NO_PADDING:
        # log_prob and loss mask are nested tensors of shape [bsz, j1]
        # for each sample, loss mask shape is [1, prompt_length + response_length]
        loss_mask = data["loss_mask"]

        log_prob_flatten = log_prob.values()
        loss_mask_flatten = loss_mask.values()

        # left-shift the loss mask by one token to align with log_prob
        loss_mask_flatten = torch.roll(loss_mask_flatten, shifts=-1, dims=0)

        # 这里计算交叉熵， 然后 取负号，最后除以 batch_num_tokens，得到平均 loss， 然后dp size 是多设备训练时的设备数量，乘上去是提前估计算出来的loss是多少
        loss = -masked_sum(log_prob_flatten, loss_mask_flatten) / batch_num_tokens * dp_size
    
    # 在有padding 模式下， 会提前有个respnse mask 用来区分prompt 和 response， 然后这里response mask 已经做移了一位
    else:
        response_mask = data["response_mask"].to(bool)
        loss = -masked_sum(log_prob, response_mask) / batch_num_tokens * dp_size

    return loss, {}

## prepare_model_inputs - 根据训练的时候是否使用带有padding数据进行训练（use remove padding），然后进行处理，在这个部分获得attention mask

> "input_ids": input_ids_rmpad,
> 
                "attention_mask": None,

                "position_ids": position_ids_rmpad,

这里input_ids_rmpad_rolled 移动后的input ids 可以用来计算loss， 然后temperature_rmpad 用来训练和评估的时候使用

> output_args["input_ids_rmpad_rolled"] = input_ids_rmpad_rolled
> 
            output_args["temperature_rmpad"] = temperature_rmpad



[automodel/transformer_impl.py (line 473)](E:/anaconda3/envs/verl/Lib/site-packages/verl/workers/engine/automodel/transformer_impl.py:473)

In [ ]:
    def prepare_model_inputs(self, micro_batch: TensorDict):
        use_remove_padding = tu.get_non_tensor_data(data=micro_batch, key="use_remove_padding", default=True)
        pad_mode = tu.get_non_tensor_data(data=micro_batch, key="pad_mode", default=DatasetPadMode.NO_PADDING)
        use_fused_kernels = tu.get_non_tensor_data(data=micro_batch, key="use_fused_kernels", default=False)
        temperature = micro_batch["temperature"]
        temperature_item = temperature
        if use_fused_kernels:
            assert not isinstance(temperature, torch.Tensor), (
                "use_fused_kernels does not support per sample temperature yet"
            )
        assert pad_mode == DatasetPadMode.NO_PADDING, f"pad_mode {pad_mode} not supported"

        multi_modal_inputs = extract_multi_modal_inputs(micro_batch.get("multi_modal_inputs", []))
        input_ids = micro_batch["input_ids"]
        position_ids = micro_batch["position_ids"]

        if not isinstance(temperature, torch.Tensor):
            temperature = torch.tensor([temperature] * input_ids.shape[0], device=input_ids.device)

        temperature = temperature.to(torch.float32)
        assert temperature.shape[0] == input_ids.shape[0]

        output_args = {}
        # use_remove_padding 是一种高级的优化策略（把长短不一的句子拼起来）。
        # NO_PADDING 是一种数据集的存储格式（数据在内存里是怎么打包的）。
        
        # 这里用去掉 padding 的方式来处理数据，主要是为了提高计算效率。因为在训练语言模型时，很多句子长度不一
        if use_remove_padding:
            temperature_rmpad = verl_F.expand_as_nested(temperature, input_ids).values()
            temperature_rmpad = temperature_rmpad.unsqueeze(0)

            if pad_mode == DatasetPadMode.NO_PADDING:
                # values扩展成一维序列，然后在前面添加一个维度给他扩展成[batch, seq_len]
                input_ids_rmpad = input_ids.values().unsqueeze(0)
                if position_ids.dim() == 3:
                    position_ids_rmpad = position_ids.values().unsqueeze(1)
                else:
                    position_ids_rmpad = position_ids.values().unsqueeze(0)
            else:
                raise NotImplementedError(f"pad_mode {pad_mode} not implemented")

            # 这里roll函数需要一个二维的， 所以前面需要给他添加一个维度
            input_ids_rmpad_rolled = torch.roll(input_ids_rmpad, shifts=-1, dims=1)

            # 把前面添加的维度去掉，变成一维序列
            input_ids_rmpad_rolled = input_ids_rmpad_rolled.squeeze(0)
            temperature_rmpad = temperature_rmpad.squeeze(0)
            output_args["input_ids_rmpad_rolled"] = input_ids_rmpad_rolled
            output_args["temperature_rmpad"] = temperature_rmpad

            model_inputs = {
                "input_ids": input_ids_rmpad,
                "attention_mask": None,
                "position_ids": position_ids_rmpad,
            }

            # For TE attention backend, pass cu_seqlens
            if self.engine_config.attn_implementation == "te":
                cu_seqlens = input_ids.offsets().to(torch.int32)
                max_seqlen = cu_seqlens.diff().max().item()
                model_inputs["qkv_format"] = "thd"
                model_inputs["cu_seqlens"] = cu_seqlens.unsqueeze(0)
                model_inputs["max_seqlen"] = max_seqlen

        # 
        else:
            if pad_mode == DatasetPadMode.NO_PADDING:
                input_ids = micro_batch["input_ids"]
                position_ids = micro_batch["position_ids"]
                loss_mask = micro_batch["loss_mask"]

                pad_token_id = tu.get_non_tensor_data(data=micro_batch, key="pad_token_id", default=0)
                batch_size = micro_batch.batch_size[0]
                seq_len_effective = input_ids.offsets().diff()
                max_seq_len = max(seq_len_effective)

                input_ids_rmpad_rolled = torch.roll(input_ids.values(), shifts=-1, dims=0)
                output_args["input_ids_rmpad_rolled"] = input_ids_rmpad_rolled
                output_args["temperature"] = temperature

                input_ids = torch.nested.to_padded_tensor(
                    input_ids, padding=pad_token_id, output_size=(batch_size, max_seq_len)
                )

                if position_ids.dim() == 3:
                    position_ids = torch.nested.to_padded_tensor(
                        position_ids, padding=0, output_size=(batch_size, 4, max_seq_len)
                    ).transpose(0, 1)
                else:
                    position_ids = torch.nested.to_padded_tensor(
                        position_ids, padding=0, output_size=(batch_size, max_seq_len)
                    )

                attention_mask_list = [torch.ones_like(t, dtype=torch.int32) for t in loss_mask]
                attention_mask = torch.nested.as_nested_tensor(attention_mask_list, layout=torch.jagged)
                attention_mask = torch.nested.to_padded_tensor(
                    attention_mask, padding=0, output_size=(batch_size, max_seq_len)
                )

                model_inputs = {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "position_ids": position_ids,
                }

            else:
                raise NotImplementedError(f"pad_mode {pad_mode} not implemented")

        extra_args = {}
        if use_fused_kernels:
            extra_args["temperature"] = temperature_item
            extra_args["return_dict"] = True

        model_inputs.update(multi_modal_inputs)
        model_inputs.update(extra_args)

        return model_inputs, output_args

# prepare_model_outputs- 把原始分数除以 Temperature 进行缩放，然后提取出预测的对数概率（Log Probs）和熵（Entropy），最后再把它们还原成嵌套张量（Nested Tensor）的格式。

1. 交叉熵 Loss（训练用）
计算逻辑：只关心正确答案
1. 把 logits ÷ Temperature，做 softmax 得到全局概率分布；
2. 只取出真实标签 label 对应的那一个概率；
3. 对这个概率做 负对数，得到 loss。
👉 目的：逼模型拉大正确词的概率，不管其他词乱不乱。

---
2. 香农熵 Entropy（看不确定性用）
计算逻辑：关心全局所有词的概率分布
1. 同样 logits ÷ Temperature + softmax 得到概率分布；
2. 遍历整个词表所有词语的概率 \(P_i\)；
3. 代入香农熵公式计算整体混乱度。
香农熵公式：
$$Entropy = -\sum_{i} P_i \log P_i$$
👉 作用：
- 熵越大：模型 犹豫不决，很多词概率差不多；
- 熵越小：模型 非常笃定，一个词概率遥遥领先

A 是 0.99，B 是 0.01
交叉熵（假设正确答案是 A）：-log(0.99) ≈ 0.01 （惩罚极小，模型很高兴）
香农熵：- (0.99 * log(0.99)) - (0.01 * log(0.01)) ≈ 0.01 + 0.046 = 0.056 （熵极低，说明模型快变成复读机了）

raw_output.logits
        +
rolled input_ids（正确答案）
        +
temperature
        ↓
每个位置正确 token 的 log_probs

In [ ]:
    def prepare_model_outputs(self, output, output_args, micro_batch: TensorDict):
        use_remove_padding = tu.get_non_tensor_data(data=micro_batch, key="use_remove_padding", default=True)
        pad_mode = tu.get_non_tensor_data(data=micro_batch, key="pad_mode", default=DatasetPadMode.NO_PADDING)
        use_fused_kernels = tu.get_non_tensor_data(data=micro_batch, key="use_fused_kernels", default=False)
        calculate_entropy = tu.get_non_tensor_data(data=micro_batch, key="calculate_entropy", default=False)

        if isinstance(output, torch.Tensor):
            from types import SimpleNamespace

            output = SimpleNamespace(logits=output)

        model_output = {}
        input_ids = micro_batch["input_ids"]

        if use_remove_padding:
            input_ids_rmpad_rolled = output_args["input_ids_rmpad_rolled"]
            temperature_rmpad = output_args["temperature_rmpad"]

            # # 如果使用了融合算子（底层算子已经帮我们算好了 log_probs 和 entropy）
            if use_fused_kernels:
                log_probs = output.log_probs.squeeze(0) # use_remove_padding = True 时，log_probs 是一个一维的张量，但是算子要求保持batch，所以目前是【1， log_probs_length】，表示去掉 padding 后的 log_probs
                entropy_rmpad = output.entropy.squeeze(0)
            else:
                logits_rmpad = output.logits.squeeze(0) # 去掉 batch 维度，变成 [Total_Tokens, Vocab_Size]
                # With TP, logits are DTensors sharded on vocab dim; gather for log_softmax.
                if isinstance(logits_rmpad, DTensor):
                    logits_rmpad = logits_rmpad.full_tensor()
                logits_rmpad = logits_rmpad / temperature_rmpad.clamp(min=1e-8).unsqueeze(-1).to(logits_rmpad.dtype)

                inplace_backward = True
                if calculate_entropy:
                    inplace_backward = False
                    
                # 计算log_prob ，这里相当于从logits里面抓取到对应位置的真实标签的概率，然后取log，最后再取负号，得到交叉熵的损失
                log_probs = logprobs_from_logits(
                    logits=logits_rmpad,
                    labels=input_ids_rmpad_rolled,
                    inplace_backward=inplace_backward,
                )

                # 计算香农熵，判断模型是否迷茫
                if calculate_entropy:
                    if not self.engine_config.entropy_checkpointing:
                        entropy_rmpad = self.compute_entropy_from_logits(logits_rmpad)
                    else:
                        entropy_rmpad = torch.utils.checkpoint.checkpoint(
                            self.compute_entropy_from_logits, logits_rmpad
                        )

            if pad_mode == DatasetPadMode.NO_PADDING:
                cu_seqlens = input_ids.offsets()
                log_probs = torch.nested.nested_tensor_from_jagged(log_probs, cu_seqlens)
                if calculate_entropy:
                    entropy = torch.nested.nested_tensor_from_jagged(entropy_rmpad, cu_seqlens)
            else:
                raise NotImplementedError(f"pad_mode {pad_mode} not implemented")

        else:
            response_length = tu.get_non_tensor_data(data=micro_batch, key="max_response_length", default=1024)
            if use_fused_kernels:
                log_probs = output.log_probs[:, -response_length - 1 : -1]
                entropy = output.entropy[:, -response_length - 1 : -1]
            else:
                logits = output.logits
                # With TP, logits are DTensors sharded on vocab dim; gather for log_softmax.
                if isinstance(logits, DTensor):
                    logits = logits.full_tensor()
                temperature = output_args["temperature"]
                temperature = temperature.unsqueeze(-1).unsqueeze(-1)
                logits = logits / temperature.clamp(min=1e-8).to(logits.dtype)

                if calculate_entropy:
                    if not self.engine_config.entropy_checkpointing:
                        entropy = verl_F.entropy_from_logits(logits)
                    else:
                        entropy = torch.utils.checkpoint.checkpoint(verl_F.entropy_from_logits, logits)

                if pad_mode == DatasetPadMode.NO_PADDING:
                    cu_seqlens = input_ids.offsets()
                    seq_lengths = cu_seqlens.diff()
                    starts = torch.zeros_like(seq_lengths, dtype=torch.int64)
                    logits = torch.nested.narrow(logits, 1, starts, seq_lengths, layout=torch.jagged)
                    logits_rmpad = torch.cat([t for t in logits.unbind()])
                    input_ids_rmpad_rolled = output_args["input_ids_rmpad_rolled"]
                    log_probs = logprobs_from_logits(logits=logits_rmpad, labels=input_ids_rmpad_rolled)
                    log_probs = torch.nested.nested_tensor_from_jagged(log_probs, cu_seqlens)
                    if calculate_entropy:
                        entropy = torch.nested.narrow(entropy, 1, starts, seq_lengths, layout=torch.jagged)
                        entropy_rmpad = torch.cat([t for t in entropy.unbind()])
                        entropy = torch.nested.nested_tensor_from_jagged(entropy_rmpad, cu_seqlens)
                else:
                    raise NotImplementedError(f"pad_mode {pad_mode} not implemented")

        model_output["log_probs"] = log_probs
        if calculate_entropy:
            model_output["entropy"] = entropy

        return model_output


In [ ]:

def forward_step(self, micro_batch: TensorDict, loss_function, forward_only):
    """Run forward pass, compute loss, and return outputs."""
    device_name = get_device_name()
    micro_batch = micro_batch.to(get_device_id())
    model_inputs, output_args = self.prepare_model_inputs(micro_batch=micro_batch)

    
    with torch.autocast(device_type=device_name, dtype=torch.bfloat16):
        raw_output = self.module(
            **model_inputs,
            use_cache=False,
        )
        # model_output 当中logits最重要
        model_output = self.prepare_model_outputs(
            output=raw_output, output_args=output_args, micro_batch=micro_batch
        )
        
        # 这里loss funcion 已经替换成了上面的sft loss
        if loss_function is not None:
            loss, metrics = loss_function(
                model_output=model_output, data=micro_batch, dp_group=self.get_data_parallel_group()
            )
        else:
            assert forward_only, "forward_only must be True when loss_function is None"
            loss = torch.tensor(1.0, device=device_name)
            metrics = {}

        output = {
            "model_output": model_output,
            "loss": loss.detach().item(),
            "metrics": metrics,
        }

        return loss, output


MultiTurnSFTDataset

→ prepare_model_inputs

→ 模型产生 raw_output.logits

→ prepare_model_outputs

→ model_output["log_probs"]

→ sft_loss

→ 得到 loss

## 梯度跟新

一个训练 batch
    ↓
拆成多个 micro-batch
    ↓
micro-batch 1 → forward_step → loss → backward
micro-batch 2 → forward_step → loss → backward
micro-batch 3 → forward_step → loss → backward
    ↓
梯度累积完成
    ↓
optimizer.step()

# 多卡梯度更新

1. 使用all reduce 进行数量上的统计
2. prepare_for_grad_accumulation([self.module]) / prepare_for_grad_accumulation([self.module]) 多卡开始计算和结束计算梯度的暗号来平均多卡上的梯度计算

In [ ]:
    def forward_backward_batch(self, data: TensorDict, loss_function: Callable, forward_only=False) -> Any:
        # 统计全局的一个 Token 总数，然后和在多少张卡上面分布式
        batch_num_tokens = data["loss_mask"].sum().to(get_device_id())
        
        # All-reduce ，可以把全部GPU上的token进行相加，然后算出总数
        # 下面函数是对 group 组张卡 GPU，执行 option op 操作，然后把值放到 batch num tokens。
        torch.distributed.all_reduce(
            batch_num_tokens, op=torch.distributed.ReduceOp.SUM, group=self.get_data_parallel_group()
        )
        tu.assign_non_tensor(data, batch_num_tokens=batch_num_tokens.item())
        tu.assign_non_tensor(data, dp_size=self.get_data_parallel_size())

        # 将整个 batch 切分成小的 microbatch
        micro_batches, indices = prepare_micro_batches(
            data=data, dp_group=self.get_data_parallel_group(), same_micro_num_in_dp=True
        )

        output_lst = []
        
        # 【准备上下文】如果是纯评估（forward_only），就开启 torch.no_grad() 省显存；否则正常记录。
        ctx = torch.no_grad() if forward_only else nullcontext()

        if not forward_only:
            # 【准备梯度累积】告诉模型：“接下来你要连续算好几个小批次，梯度先攒着，别急着更新参数。”
            prepare_for_grad_accumulation([self.module])

            # Set MoE aux loss backward scale to counteract FSDP's gradient allreduce.
            if self.engine_config.ep_size > 1:
                from nemo_automodel.components.moe.megatron.moe_utils import MoEAuxLossAutoScaler

                MoEAuxLossAutoScaler.main_loss_backward_scale = torch.tensor(
                    float(get_dp_group_size(self.device_mesh, include_cp=True))
                )

        num_micro_batches = len(micro_batches)
        for i, micro_batch in enumerate(micro_batches):
            # 如果现在是训练模式，并且当前已经是【最后一个】小批次了：
        # 告诉模型：“这是最后一块数据了！马上准备执行 All-Reduce，把大家攒的梯度汇总同步吧！”
            if not forward_only and i == num_micro_batches - 1:
                prepare_for_final_backward([self.module])

            with ctx:
                 # 1. 前向传播：喂数据，算出 Loss。（这就是我们之前看的那个加工车间）
                loss, meta_info = self.forward_step(micro_batch, loss_function=loss_function, forward_only=forward_only)
                if not forward_only:# 2. 反向传播：如果不是纯评估，就根据 Loss 算出梯度，并悄悄累积到模型参数上。
                    loss.backward()
            output_lst.append(meta_info)

        return postprocess_batch_func(output_lst=output_lst, indices=indices, data=data)